#  Kazakh Punctuation Restoration - XLM-RoBERTa + CRF
1. Запускай секции: 1 → 2 → 3 → 4 → 9 → 10 → 14
2. Готовый файл: `/kaggle/working/submission.csv`

## 1. Install

In [2]:
!pip install transformers==4.47.0 datasets seqeval accelerate -q
!pip install pytorch-crf scipy -q
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 55.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.7 MB/s eta 0:00:00:00:01


## 2. Imports

In [9]:
import os, re, random, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.optimize import minimize
from scipy.special import softmax

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


## 3. Config

In [22]:
MODEL_NAME = 'xlm-roberta-large'

LABEL2ID     = {'O': 0, 'COMMA': 1, 'PERIOD': 2, 'QUESTION': 3}
ID2LABEL     = {v: k for k, v in LABEL2ID.items()}
IGNORE_INDEX = -100

MAX_LEN    = 512
BATCH_SIZE = 32
GRAD_ACCUM = 2       
LR         = 1e-5
EPOCHS     = 5

N_MULTIDOMAIN = 200_000  # текстов из multidomain датасета
N_WIKI        =  50_000  # текстов из Wikipedia KK
MAX_TRAIN     = 120_000 


CHECKPOINT_PT  = '/kaggle/input/kaz-punct-v5-crf/model.pt'   
CHECKPOINT_DIR = '/kaggle/input/kaz-punct-v5-crf'           

CHECKPOINT_PT  = '/kaggle/input/datasets/mausymzhanmakazhan/modelkaz/FASHION/model.pt'
CHECKPOINT_DIR = '/kaggle/input/datasets/mausymzhanmakazhan/modelkaz/FASHION'

SAVE_DIR  = '/kaggle/working/kaz_punct_v5_crf'  
DATA_DIR  = Path('data');  DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path('model'); MODEL_DIR.mkdir(exist_ok=True)

print(f'Config OK | Model: {MODEL_NAME} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}')

Config OK | Model: xlm-roberta-large | Effective batch: 64


## 4. Load competition files

In [12]:
test_df = pd.read_csv('/kaggle/input/competitions/kaz-punct-hackathon/test.csv')
train_example = pd.read_csv('/kaggle/input/competitions/kaz-punct-hackathon/train_example.csv')
print(f'Train example: {len(train_example)} rows')
print(f'Test:          {len(test_df)} rows')
train_example.head(2)

Train example: 500 rows
Test:          3552 rows


,id,input_text,labels
0,kzp_train_0001,ерден ердің қаупі бар,O O O PERIOD
1,kzp_train_0002,ақылдының ақылы сарқылмайтын көлмен тең,O O O O PERIOD


## 5. Preprocessing

In [ ]:
KAZ_PUNCT = re.compile(r'[«»"\(\)\[\]\{\};:\-–—/\\]')

def strip_and_label(text: str):
    """Убирает пунктуацию из текста и записывает где она стояла как метки."""
    tokens = text.split()
    clean, labels = [], []
    for tok in tokens:
        tok = KAZ_PUNCT.sub('', tok).strip()
        if not tok:
            continue
        if tok.endswith('?'):
            label, tok = 'QUESTION', tok[:-1]
        elif tok.endswith(('!', '.', '…')):
            label, tok = 'PERIOD', tok.rstrip('!.…')
        elif tok.endswith(','):
            label, tok = 'COMMA', tok[:-1]
        else:
            label = 'O'
        tok = tok.lower().strip()
        if tok:
            clean.append(tok)
            labels.append(label)
    return clean, labels


def texts_to_rows(texts, max_words=50, min_words=8):
    """Конвертирует тексты в строки датасета (input_text, labels)."""
    rows = []
    sent_splitter = re.compile(r'(?<=[.!?])\s+')
    for text in texts:
        if not isinstance(text, str) or len(text) < 20:
            continue
        buf_tok, buf_lbl = [], []
        for sent in sent_splitter.split(text.strip()):
            tokens, labels = strip_and_label(sent)
            if not tokens:
                continue
            if len(buf_tok) + len(tokens) > max_words and len(buf_tok) >= min_words:
                rows.append({'input_text': ' '.join(buf_tok),
                             'labels':     ' '.join(buf_lbl)})
                buf_tok, buf_lbl = [], []
            buf_tok.extend(tokens)
            buf_lbl.extend(labels)
        if len(buf_tok) >= min_words:
            rows.append({'input_text': ' '.join(buf_tok),
                         'labels':     ' '.join(buf_lbl)})
    return rows


def is_quality_row(labels_str):
    """Фильтр качества: строка должна содержать COMMA и быть достаточно пунктуированной."""
    labels = labels_str.split()
    if 'COMMA' not in labels:
        return False
    return labels.count('O') / len(labels) < 0.95

print('Preprocessing ready')

## 6. Build training data
> Два источника: multidomain-kazakh-dataset + Kazakh Wikipedia

In [ ]:
def stream_dataset(hf_path, n_target, text_col='text',
                   batch_size=10_000, split_paragraphs=False, **load_kwargs):
    ds = load_dataset(hf_path, split='train', streaming=True, **load_kwargs)
    ds = ds.shuffle(seed=SEED, buffer_size=50_000)
    rows, batch, collected = [], [], 0
    for row in ds:
        text = row.get(text_col, '')
        if 'predicted_language' in row and row['predicted_language'] != 'kaz':
            continue
        if 'contains_kaz_symbols' in row and row['contains_kaz_symbols'] != 1:
            continue
        if not isinstance(text, str) or len(text) < 30:
            continue
        if split_paragraphs:
            batch.extend([p.strip() for p in text.split('\n') if len(p.strip()) > 30])
        else:
            batch.append(text)
        if len(batch) >= batch_size:
            rows.extend(texts_to_rows(batch))
            collected += len(batch)
            batch = []
            print(f'  {collected:,} texts -> {len(rows):,} rows')
        if collected >= n_target:
            break
    if batch:
        rows.extend(texts_to_rows(batch))
    return rows


all_rows = []

print('=== Source 1: multidomain-kazakh-dataset ===')
all_rows.extend(stream_dataset(
    'kz-transformers/multidomain-kazakh-dataset', n_target=N_MULTIDOMAIN
))
print(f'After source 1: {len(all_rows):,}')

print('\n=== Source 2: Kazakh Wikipedia ===')
try:
    all_rows.extend(stream_dataset(
        'wikimedia/wikipedia', n_target=N_WIKI,
        text_col='text', split_paragraphs=True, name='20231101.kk',
    ))
    print(f'After source 2: {len(all_rows):,}')
except Exception as e:
    print(f'Wikipedia failed: {e} — continuing without it')

# Добавляем золотые примеры из соревнования
for _, r in train_example.iterrows():
    all_rows.append({'input_text': r['input_text'], 'labels': r['labels']})

print(f'Total: {len(all_rows):,}')

## 7. Quality filter & split

In [ ]:
df = pd.DataFrame(all_rows).drop_duplicates('input_text').reset_index(drop=True)
print(f'Before filter: {len(df):,}')

df_quality = df[df['labels'].apply(is_quality_row)].reset_index(drop=True)
df_other   = df[~df['labels'].apply(is_quality_row)]
n_other    = min(len(df_quality) // 2, len(df_other))
df = pd.concat([df_quality, df_other.sample(n_other, random_state=SEED)])
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'After filter: {len(df):,}')

if len(df) > MAX_TRAIN + 5000:
    df = df[:MAX_TRAIN + 5000]

all_labels = ' '.join(df['labels']).split()
cnt = Counter(all_labels)
total = sum(cnt.values())
print('Label distribution:')
for lbl, c in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f'  {lbl:10s}: {c:>10,}  ({100*c/total:.1f}%)')

val_size = min(5000, int(len(df) * 0.05))
val_df   = df[:val_size]
train_df = df[val_size:]
print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}')

## 8. Tokenizer & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size:,}')

In [ ]:
def encode_row(words, labels, tokenizer, max_len=MAX_LEN):
    enc = tokenizer(
        words, is_split_into_words=True,
        max_length=max_len, truncation=True, padding=False,
    )
    word_ids, label_ids, prev_word = enc.word_ids(), [], None
    for wid in word_ids:
        if wid is None:
            label_ids.append(IGNORE_INDEX)
        elif wid != prev_word:
            label_ids.append(LABEL2ID[labels[wid]] if wid < len(labels) else IGNORE_INDEX)
        else:
            label_ids.append(IGNORE_INDEX)
        prev_word = wid
    enc['labels'] = label_ids
    return enc


class PunctDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.samples = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc='Encoding'):
            words  = row['input_text'].split()
            labels = row['labels'].split()
            if len(words) != len(labels):
                continue
            self.samples.append(encode_row(words, labels, tokenizer, max_len))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        return {k: torch.tensor(v) for k, v in self.samples[idx].items()}


print('Building datasets...')
train_dataset = PunctDataset(train_df, tokenizer)
val_dataset   = PunctDataset(val_df,   tokenizer)
print(f'Train: {len(train_dataset):,}  |  Val: {len(val_dataset):,}')

## 9. Model — XLM-RoBERTa + CRF

In [23]:
class BertCRFClassifier(nn.Module):
    """XLM-RoBERTa encoder + linear head + CRF decoder."""
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.linear  = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.crf     = CRF(num_labels, batch_first=True)
        self.encoder.gradient_checkpointing_enable()

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        hidden = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask,
        ).last_hidden_state
        logits = self.linear(self.dropout(hidden))
        loss = None
        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == IGNORE_INDEX] = 0
            crf_mask = attention_mask.bool()
            loss = -self.crf(logits, safe_labels, mask=crf_mask, reduction='mean')
        return TokenClassifierOutput(loss=loss, logits=logits)


# ── Вариант A: создать модель с нуля (для обучения) ──────────────────────────
#model = BertCRFClassifier(MODEL_NAME, num_labels=len(LABEL2ID)).to(device)

# ── Вариант B: загрузить сохранённую модель (только инференс) ────────────────
# Раскомментируй эти строки если модель уже обучена и сохранена как Kaggle Dataset
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR)  # ← CHANGE в Config
model = BertCRFClassifier(MODEL_NAME, num_labels=len(LABEL2ID)).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PT, map_location=device))  # ← CHANGE в Config
model.eval()
print('Checkpoint loaded!')

print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

Checkpoint loaded!
Parameters: 559.9M


## 10. Metrics

In [25]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    y_true, y_pred = [], []
    for pred_row, label_row in zip(preds, labels):
        for p, l in zip(pred_row, label_row):
            if l == IGNORE_INDEX: continue
            y_true.append(ID2LABEL[l])
            y_pred.append(ID2LABEL[p])
    score = f1_score(y_true, y_pred,
                     labels=['COMMA', 'PERIOD', 'QUESTION'],
                     average='macro', zero_division=0)
    per = f1_score(y_true, y_pred,
                   labels=['COMMA', 'PERIOD', 'QUESTION'],
                   average=None, zero_division=0)
    return {'macro_f1':    round(score,  4),
            'f1_comma':    round(per[0], 4),
            'f1_period':   round(per[1], 4),
            'f1_question': round(per[2], 4)}

## 11. Training

In [ ]:
import shutil
shutil.rmtree('model', ignore_errors=True)
os.makedirs('model', exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = str(MODEL_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 1,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'macro_f1',
    greater_is_better           = True,
    logging_steps               = 200,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    report_to                   = 'none',
    seed                        = SEED,
)

collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    data_collator    = collator,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## 12. Save model

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), f'{SAVE_DIR}/model.pt')
tokenizer.save_pretrained(SAVE_DIR)
print(f'Saved! Size: {os.path.getsize(f"{SAVE_DIR}/model.pt")/1e6:.0f} MB')
print(f'Files: {os.listdir(SAVE_DIR)}')
print('Download via Output panel on the right ->')

## 13. Evaluate on val set

In [ ]:
model.eval()
all_logits_eval, all_labels_eval = [], []

loader = DataLoader(
    val_dataset, batch_size=32,
    collate_fn=DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)
)
with torch.no_grad():
    for batch in tqdm(loader, desc='Evaluating'):
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = model(**batch)
        all_logits_eval.append(out.logits.cpu().numpy())
        all_labels_eval.append(batch['labels'].cpu().numpy())

y_true, y_pred = [], []
for logits, labels in zip(all_logits_eval, all_labels_eval):
    preds = np.argmax(logits, axis=-1)
    for pred_row, label_row in zip(preds, labels):
        for p, l in zip(pred_row, label_row):
            if l == IGNORE_INDEX: continue
            y_true.append(ID2LABEL[l])
            y_pred.append(ID2LABEL[p])

print(classification_report(y_true, y_pred,
      labels=['COMMA', 'PERIOD', 'QUESTION'], digits=4))
macro = f1_score(y_true, y_pred, labels=['COMMA','PERIOD','QUESTION'],
                 average='macro', zero_division=0)
print(f'Macro F1: {macro:.4f}')

## 14. 🎯 SOLUTION — Inference & Submission
> Эта секция генерирует финальный `submission.csv`  
> Использует Sliding Window + CRF Viterbi decode + постобработку казахских частиц

In [26]:
MODEL_NAME   = 'xlm-roberta-large'
LABEL2ID     = {'O': 0, 'COMMA': 1, 'PERIOD': 2, 'QUESTION': 3}
ID2LABEL     = {v: k for k, v in LABEL2ID.items()}
IGNORE_INDEX = -100
MAX_LEN      = 512

TEST_CSV   = '/kaggle/input/competitions/kaz-punct-hackathon/test.csv'              
MODEL_PT   = '/kaggle/input/datasets/mausymzhanmakazhan/modelkaz/FASHION/model.pt'        
SUBMISSION = '/kaggle/working/submission.csv'                  

CHECKPOINT_PT  = '/kaggle/input/datasets/mausymzhanmakazhan/modelkaz/FASHION/model.pt'
CHECKPOINT_DIR = '/kaggle/input/datasets/mausymzhanmakazhan/modelkaz/FASHION'

test_df = pd.read_csv(TEST_CSV)
print(f'Test: {len(test_df)} rows')

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-large')
model     = BertCRFClassifier(MODEL_NAME, num_labels=len(LABEL2ID)).to(device)
model.load_state_dict(torch.load(MODEL_PT, map_location=device))
model.eval()
print(f'Model loaded! {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

QUESTION_PARTICLES = {'ма','ме','ба','бе','па','пе','ше','ші'}

def postprocess(words, pred_labels):
    """Форсирует QUESTION для казахских вопросительных частиц."""
    labels = pred_labels.copy()
    for i, word in enumerate(words):
        if word.lower() in QUESTION_PARTICLES:
            labels[i] = 'QUESTION'
    return labels


@torch.no_grad()
def predict_sliding(words, stride=100):
    """Sliding window inference с CRF Viterbi decode."""
    model.eval()
    n         = len(words)
    win_size  = MAX_LEN // 2
    logit_sum = torch.zeros(n, len(LABEL2ID))
    logit_cnt = torch.zeros(n)

    starts = list(range(0, n, stride))
    if not starts or starts[-1] + win_size < n:
        starts.append(max(0, n - win_size))

    for start in starts:
        chunk = words[start:start + win_size]
        enc   = tokenizer(chunk, is_split_into_words=True,
                          max_length=MAX_LEN, truncation=True,
                          return_tensors='pt').to(device)
        logits   = model(**enc).logits[0].cpu().float()
        word_ids = enc.word_ids()
        seen = set()
        for tok_i, wid in enumerate(word_ids):
            if wid is None or wid in seen: continue
            gid = start + wid
            if gid < n:
                logit_sum[gid] += logits[tok_i]
                logit_cnt[gid] += 1
            seen.add(wid)

    avg     = (logit_sum / logit_cnt.clamp(min=1).unsqueeze(-1)).unsqueeze(0).to(device)
    mask    = torch.ones(1, n, dtype=torch.bool).to(device)
    decoded = model.crf.decode(avg, mask=mask)[0]
    return [ID2LABEL[p] for p in decoded]


# ── Predict ───────────────────────────────────────────────────────────────────
results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    words  = row['input_text'].split()
    labels = postprocess(words, predict_sliding(words))
    assert len(labels) == len(words), f'Length mismatch on {row["id"]}'
    results.append({'id': row['id'], 'labels': ' '.join(labels)})

submission = pd.DataFrame(results)

# ── Validate ──────────────────────────────────────────────────────────────────
valid_labels = {'O', 'COMMA', 'PERIOD', 'QUESTION'}
for _, row in test_df.iterrows():
    exp = len(row['input_text'].split())
    act = len(submission.loc[submission['id']==row['id'], 'labels'].values[0].split())
    assert exp == act, f'MISMATCH {row["id"]}'
for _, row in submission.iterrows():
    for lbl in row['labels'].split():
        assert lbl in valid_labels, f'Invalid label: {lbl}'

pred_all = ' '.join(submission['labels']).split()
pred_cnt = Counter(pred_all)
print('Prediction distribution:')
for lbl, c in sorted(pred_cnt.items(), key=lambda x: -x[1]):
    print(f'  {lbl:10s}: {c:>8,}  ({100*c/len(pred_all):.1f}%)')
print('\nAll checks passed!')

submission.to_csv(SUBMISSION, index=False)
print(f'Saved -> {SUBMISSION}')

Test: 3552 rows


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded! 559.9M params


Predicting:   0%|          | 0/3552 [00:00<?, ?it/s]

Prediction distribution:
  O         :   72,517  (85.4%)
  PERIOD    :    6,978  (8.2%)
  COMMA     :    5,038  (5.9%)
  QUESTION  :      335  (0.4%)

All checks passed!
Saved -> /kaggle/working/submission.csv


In [21]:
MODEL_NAME   = 'xlm-roberta-large'
LABEL2ID     = {'O': 0, 'COMMA': 1, 'PERIOD': 2, 'QUESTION': 3}
ID2LABEL     = {v: k for k, v in LABEL2ID.items()}
IGNORE_INDEX = -100
MAX_LEN      = 512

TEST_CSV   = '/kaggle/input/competitions/kaz-punct-hackathon/test.csv'  
HF_MODEL   = 'maussym/kaz-punct-v5-crf'                                 
SUBMISSION = '/kaggle/working/submission.csv'                            


test_df = pd.read_csv(TEST_CSV)
print(f'Test: {len(test_df)} rows')

from huggingface_hub import hf_hub_download

tokenizer  = AutoTokenizer.from_pretrained('xlm-roberta-large')
model      = BertCRFClassifier(MODEL_NAME, num_labels=len(LABEL2ID)).to(device)
model_path = hf_hub_download(repo_id=HF_MODEL, filename='model.pt')
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f'Model loaded from {HF_MODEL}')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

QUESTION_PARTICLES = {'ма','ме','ба','бе','па','пе','ше','ші'}

def postprocess(words, pred_labels):
    """Форсирует QUESTION для казахских вопросительных частиц."""
    labels = pred_labels.copy()
    for i, word in enumerate(words):
        if word.lower() in QUESTION_PARTICLES:
            labels[i] = 'QUESTION'
    return labels


@torch.no_grad()
def predict_sliding(words, stride=100):
    """Sliding window inference с CRF Viterbi decode."""
    model.eval()
    n         = len(words)
    win_size  = MAX_LEN // 2
    logit_sum = torch.zeros(n, len(LABEL2ID))
    logit_cnt = torch.zeros(n)

    starts = list(range(0, n, stride))
    if not starts or starts[-1] + win_size < n:
        starts.append(max(0, n - win_size))

    for start in starts:
        chunk = words[start:start + win_size]
        enc   = tokenizer(chunk, is_split_into_words=True,
                          max_length=MAX_LEN, truncation=True,
                          return_tensors='pt').to(device)
        logits   = model(**enc).logits[0].cpu().float()
        word_ids = enc.word_ids()
        seen = set()
        for tok_i, wid in enumerate(word_ids):
            if wid is None or wid in seen: continue
            gid = start + wid
            if gid < n:
                logit_sum[gid] += logits[tok_i]
                logit_cnt[gid] += 1
            seen.add(wid)

    avg     = (logit_sum / logit_cnt.clamp(min=1).unsqueeze(-1)).unsqueeze(0).to(device)
    mask    = torch.ones(1, n, dtype=torch.bool).to(device)
    decoded = model.crf.decode(avg, mask=mask)[0]
    return [ID2LABEL[p] for p in decoded]


results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Predicting'):
    words  = row['input_text'].split()
    labels = postprocess(words, predict_sliding(words))
    assert len(labels) == len(words), f'Length mismatch on {row["id"]}'
    results.append({'id': row['id'], 'labels': ' '.join(labels)})

submission = pd.DataFrame(results)

valid_labels = {'O', 'COMMA', 'PERIOD', 'QUESTION'}
for _, row in test_df.iterrows():
    exp = len(row['input_text'].split())
    act = len(submission.loc[submission['id']==row['id'], 'labels'].values[0].split())
    assert exp == act, f'MISMATCH {row["id"]}'
for _, row in submission.iterrows():
    for lbl in row['labels'].split():
        assert lbl in valid_labels, f'Invalid label: {lbl}'

pred_all = ' '.join(submission['labels']).split()
pred_cnt = Counter(pred_all)
print('Prediction distribution:')
for lbl, c in sorted(pred_cnt.items(), key=lambda x: -x[1]):
    print(f'  {lbl:10s}: {c:>8,}  ({100*c/len(pred_all):.1f}%)')
print('\nAll checks passed!')

submission.to_csv(SUBMISSION, index=False)
print(f'Saved -> {SUBMISSION}')

Test: 3552 rows
Model loaded from maussym/kaz-punct-v5-crf
Parameters: 559.9M


Predicting:   0%|          | 0/3552 [00:00<?, ?it/s]

Prediction distribution:
  O         :   72,517  (85.4%)
  PERIOD    :    6,978  (8.2%)
  COMMA     :    5,038  (5.9%)
  QUESTION  :      335  (0.4%)

All checks passed!
Saved -> /kaggle/working/submission.csv
